# End-to-End NLP Project

# Text Preprocessing

## Project

Email Spam Detection using Machine Learning

## Objective

The goal of this notebook is to transform raw SMS messages into a clean and structured format suitable for Natural Language Processing (NLP) and Machine Learning.

The preprocessing pipeline developed here will later be integrated into the production project inside the `src/components/data_transformation.py` module.

## Author

Aneesh Jose

# Problem Statement

Raw text cannot be directly used by machine learning algorithms.

Messages often contain:

- Uppercase and lowercase letters
- Punctuation
- Numbers
- Special symbols
- Common words (stopwords)

These elements must be cleaned and standardized before converting the text into numerical features such as Bag of Words or TF-IDF.

This notebook builds a complete text preprocessing pipeline step by step.

In [1]:
# ==========================================
# Import Required Libraries
# ==========================================

import string

import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ANEES\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ANEES\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ANEES\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## Why are these libraries required?

- Pandas is used to load and manipulate the dataset.
- NLTK provides NLP utilities such as tokenization and stopword removal.
- PorterStemmer reduces words to their root form.
- String provides punctuation characters that will be removed during preprocessing.

In [3]:
df = pd.read_csv("../artifacts/data_ingestion/spam.csv")

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Load Dataset

The standardized dataset produced during the Data Ingestion stage is loaded.

This ensures that all preprocessing is performed on the validated dataset rather than the original raw file.

In [4]:
sample_text = df["message"][2]

sample_text

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"

## Why use a sample message?

Instead of preprocessing the entire dataset immediately, we first apply every transformation to a single message.

This makes it easier to understand the effect of each preprocessing step before applying it to thousands of messages.

# Step 1 - Convert Text to Lowercase

Machine learning models treat:

Free

FREE

free

as three different words.

Converting everything to lowercase ensures that all variations are treated as the same token.

In [5]:
sample_text.lower()

"free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005. text fa to 87121 to receive entry question(std txt rate)t&c's apply 08452810075over18's"

# Step 2 - Tokenization

Tokenization splits a sentence into individual words (tokens).

These tokens become the basic units that NLP models learn from.

In [6]:
tokens = word_tokenize(sample_text.lower())

tokens

['free',
 'entry',
 'in',
 '2',
 'a',
 'wkly',
 'comp',
 'to',
 'win',
 'fa',
 'cup',
 'final',
 'tkts',
 '21st',
 'may',
 '2005.',
 'text',
 'fa',
 'to',
 '87121',
 'to',
 'receive',
 'entry',
 'question',
 '(',
 'std',
 'txt',
 'rate',
 ')',
 't',
 '&',
 'c',
 "'s",
 'apply',
 '08452810075over18',
 "'s"]

# Step 3 - Remove Special Characters

Special characters such as punctuation generally do not contribute meaningful information for spam classification.

Only alphanumeric tokens are retained.

In [9]:
clean_tokens = [token for token in tokens if token.isalnum()]

clean_tokens

['free',
 'entry',
 'in',
 '2',
 'a',
 'wkly',
 'comp',
 'to',
 'win',
 'fa',
 'cup',
 'final',
 'tkts',
 '21st',
 'may',
 'text',
 'fa',
 'to',
 '87121',
 'to',
 'receive',
 'entry',
 'question',
 'std',
 'txt',
 'rate',
 't',
 'c',
 'apply',
 '08452810075over18']

# Step 4 - Remove Stopwords

Stopwords are extremely common words such as:

- the
- is
- are
- of

These words appear in almost every sentence and usually provide little information for distinguishing spam from ham.

However, words like "not" should be considered carefully because removing them can change the meaning of a sentence. In this spam detection project, we follow the standard English stopword list while being aware of this trade-off.

In [10]:
stop_words = set(stopwords.words("english"))

filtered_tokens = [token for token in clean_tokens if token not in stop_words]

filtered_tokens

['free',
 'entry',
 '2',
 'wkly',
 'comp',
 'win',
 'fa',
 'cup',
 'final',
 'tkts',
 '21st',
 'may',
 'text',
 'fa',
 '87121',
 'receive',
 'entry',
 'question',
 'std',
 'txt',
 'rate',
 'c',
 'apply',
 '08452810075over18']

# Step 5 - Stemming

Different forms of the same word often convey the same meaning.

Examples:

play

playing

played

player

Stemming reduces these words to a common root.

This reduces vocabulary size and improves model generalization.

In [11]:
ps = PorterStemmer()

stemmed_tokens = [ps.stem(token) for token in filtered_tokens]

stemmed_tokens

['free',
 'entri',
 '2',
 'wkli',
 'comp',
 'win',
 'fa',
 'cup',
 'final',
 'tkt',
 '21st',
 'may',
 'text',
 'fa',
 '87121',
 'receiv',
 'entri',
 'question',
 'std',
 'txt',
 'rate',
 'c',
 'appli',
 '08452810075over18']

In [ ]:
# def transform_text(message):
#     filtered_tokens=[ps.stem(token)
#                      for token in word_tokenize(message.lower())
#                        if token.isalnum() and token not in stop_words]
#     return filtered_tokens

In [24]:
def transform_text(message):
    
    # Step 1 - Convert to lowercase
    message = message.lower()

    # Step 2 - Tokenize
    tokens = word_tokenize(message)

    # Step 3 - Keep only alphanumeric tokens
    tokens = [
        token
        for token in tokens
        if token.isalnum()
    ]

    # Step 4 - Remove stopwords
    tokens = [
        token
        for token in tokens
        if token not in stop_words
    ]

    # Step 5 - Apply stemming
    tokens = [
        ps.stem(token)
        for token in tokens
    ]

    # Step 6 - Join tokens back into a sentence
    return " ".join(tokens)

In [25]:
df["transformed_text"] = df["message"].apply(transform_text)

In [26]:
df[
    [
        "message",
        "transformed_text"
    ]
].head()

,message,transformed_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


- Lowercase conversion standardizes text.
- Tokenization separates words.
- Special characters were removed.
- Stopwords reduced noise.
- Stemming reduced vocabulary size.
- The transformed text is now ready for vectorization.

The preprocessing pipeline successfully converted raw SMS messages into clean text suitable for machine learning.

The processed dataset will be used in the next notebook to build numerical representations using:

- Bag of Words
- TF-IDF

before training multiple machine learning models.

In [30]:
sample_message = df.loc[2, "message"]

print("Original Message:")
print(sample_message)
print("\nProcessed Message:")
print(transform_text(sample_message))

Original Message:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's

Processed Message:
free entri 2 wkli comp win fa cup final tkt 21st may text fa 87121 receiv entri question std txt rate c appli 08452810075over18


In [31]:
from pathlib import Path

output_dir = Path("../artifacts/data_transformation")

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [32]:
output_file = output_dir / "spam_cleaned.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Dataset saved to: {output_file}")

Dataset saved to: ..\artifacts\data_transformation\spam_cleaned.csv


# Key Insights

The following preprocessing operations were successfully performed:

- Converted all text to lowercase.
- Tokenized each message into individual words.
- Removed punctuation and special characters.
- Removed English stopwords.
- Applied Porter Stemming.
- Generated a cleaned version of every SMS message.
- Saved the processed dataset for feature extraction.

The dataset is now ready for numerical vectorization using techniques such as Bag of Words and TF-IDF.

# Conclusion

The text preprocessing pipeline successfully transformed raw SMS messages into a standardized format suitable for Natural Language Processing.

The processed text will be used in the next notebook to convert textual information into numerical feature vectors using:

- Bag of Words (BoW)
- TF-IDF

These numerical representations will then be used to train and evaluate multiple machine learning models for spam detection.